# Brazil 2026 — States, Congress and the Senate
### A dossier for each of the 27 federal units, and what the 513+54 seats mean

This notebook goes from the national aggregate to the territory. It has four parts:

1. **Congress as a whole** — the 513 Chamber seats and the 54 Senate seats renewed in 2026, by party and by federation, and the 27 governorships.
2. **The Senate** — who won, with how many votes, and how the two-seat-per-state rule shaped the result.
3. **The Chamber and the state assemblies** — delegation sizes, vote-to-seat conversion by state, the vote-pullers, and the internal composition of each legislature.
4. **A dossier for each state** — governor, senators, Chamber delegation, assembly composition, turnout, and an interactive municipal map of the governor race.

Method notes that apply throughout:

* Winners are identified from the final situation recorded by the TSE (DS_SIT_TOT_TURNO = ELEITO, ELEITO POR QP, ELEITO POR MÉDIA). This includes the deputies elected by party quotient (QP) and by average (média).
* **Federations** (PT/PCdoB/PV, UNIÃO/PP, PSDB/Cidadania, PRD/Solidariedade, PSOL/REDE) compete as a single party in proportional elections, so their seats are pooled in the federation views.
* Senate shares use valid Senate votes as denominator; with two votes per voter, the national total is ~207 million votes for ~125 million voters.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# locate the project root (works whichever directory Jupyter was started from)
_p = Path.cwd()
while not (_p / "00_data_preprocessing_validation.ipynb").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, Markdown, display

import types, nbformat

_lib_path = _p / "00_data_preprocessing_validation.ipynb"
_lib_src = "\n".join(
    c.source for c in nbformat.read(str(_lib_path), as_version=4).cells
    if c.cell_type == "code" and "library" in (c.metadata.get("tags") or [])
)
elib = types.ModuleType("elib")
elib.__dict__["__file__"] = str(_lib_path)
exec(compile(_lib_src, str(_lib_path), "exec"), elib.__dict__)

if getattr(elib, "LIB_VERSION", 0) < 2 or not hasattr(elib, "enable_plotting"):
    print("WARNING: 00_data_preprocessing_validation.ipynb is older than this notebook.")
    print("         Copy the whole repository again (all four notebooks together).")
    try:
        import nbformat  # noqa: F401
        pio.renderers.default = "plotly_mimetype"
    except Exception:
        pio.renderers.default = "notebook_connected"
else:
    elib.enable_plotting()
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
con = elib.db("2GB")
print("ready")

ready


In [2]:
# ---------- core aggregates ----------
gov = con.execute(r'''
WITH g AS (
  SELECT SG_UF, NR_TURNO, SQ_CANDIDATO, NM_CANDIDATO, NM_URNA_CANDIDATO, SG_PARTIDO, sum(VOTOS) v
  FROM cand WHERE DS_CARGO = 'Governador' GROUP BY ALL),
t AS (SELECT SG_UF, NR_TURNO, sum(v) tot FROM g GROUP BY 1, 2),
r AS (SELECT g.*, t.tot, 100.0 * g.v / t.tot pct,
             row_number() OVER (PARTITION BY SG_UF, NR_TURNO ORDER BY g.v DESC) rk
      FROM g JOIN t USING (SG_UF, NR_TURNO)),
w AS (SELECT SG_UF, max(NR_TURNO) turno FROM g GROUP BY 1)
SELECT r.* FROM r JOIN w ON r.SG_UF = w.SG_UF AND r.NR_TURNO = w.turno
''').df()

sen = con.execute(r'''
SELECT SG_UF, SQ_CANDIDATO, NM_CANDIDATO, NM_URNA_CANDIDATO, SG_PARTIDO, TP_AGREMIACAO,
       sum(VOTOS) votos, any_value(DS_SIT_TOT_TURNO) situacao
FROM cand WHERE DS_CARGO = 'Senador' GROUP BY 1, 2, 3, 4, 5, 6
''').df()
sen["share"] = 100 * sen.votos / sen.groupby("SG_UF").votos.transform("sum")
sen_win = sen[sen.situacao == "ELEITO"].copy()

ch = con.execute(r'''
SELECT SG_UF, SQ_CANDIDATO, NM_CANDIDATO, NM_URNA_CANDIDATO, SG_PARTIDO, sum(VOTOS) votos,
       any_value(DS_SIT_TOT_TURNO) situacao, any_value(TP_AGREMIACAO) agr
FROM cand WHERE DS_CARGO = 'Deputado Federal' GROUP BY 1, 2, 3, 4, 5
''').df()
ch_win = ch[ch.situacao.str.startswith("ELEITO")].copy()

asm = con.execute(r'''
SELECT SG_UF, DS_CARGO, SQ_CANDIDATO, NM_CANDIDATO, NM_URNA_CANDIDATO, SG_PARTIDO, sum(VOTOS) votos,
       any_value(DS_SIT_TOT_TURNO) situacao
FROM cand WHERE DS_CARGO IN ('Deputado Estadual', 'Deputado Distrital') GROUP BY 1, 2, 3, 4, 5, 6
''').df()
asm_win = asm[asm.situacao.str.startswith("ELEITO")].copy()

turn = con.execute(r'''
SELECT SG_UF, sum(QT_APTOS) aptos, sum(QT_COMPARECIMENTO) comp,
       100.0 * sum(QT_COMPARECIMENTO) / sum(QT_APTOS) part
FROM det WHERE CD_CARGO = 3 AND NR_TURNO = 1 GROUP BY 1
''').df()
turn = elib.add_uf_meta(turn)

print(f"governor rows {len(gov)} | senate winners {len(sen_win)} | chamber winners {len(ch_win)} | assembly winners {len(asm_win)}")

governor rows 153 | senate winners 54 | chamber winners 513 | assembly winners 1059


## 1. The shape of the new Congress

The Chamber elected in 2026 has **513 deputies from 20 parties**. Measured by the party label alone that is a fragmented chamber; measured by **federation** — the unit that actually competes in proportional elections — it has 15 competitors, and two of them (PL and the PT/PCdoB/PV federation) hold 209 seats between them.

In [3]:
ch_party = ch_win.groupby("SG_PARTIDO", as_index=False).size().rename(columns={"size": "seats"})
ch_party["share"] = 100 * ch_party.seats / ch_party.seats.sum()
ch_party = ch_party.sort_values("seats", ascending=False)

fed = {"PT": "PT/PCdoB/PV", "PCDOB": "PT/PCdoB/PV", "PV": "PT/PCdoB/PV",
       "UNIÃO": "UNIÃO/PP", "PP": "UNIÃO/PP", "PSDB": "PSDB/Cidadania", "CIDADANIA": "PSDB/Cidadania",
       "PRD": "PRD/Solidariedade", "SOLIDARIEDADE": "PRD/Solidariedade",
       "PSOL": "PSOL/REDE", "REDE": "PSOL/REDE"}
ch_party["federation"] = ch_party.SG_PARTIDO.map(fed).fillna(ch_party.SG_PARTIDO)

fig = go.Figure(go.Bar(
    x=ch_party.seats, y=ch_party.SG_PARTIDO, orientation="h",
    marker=dict(color=[elib.party_color(p) for p in ch_party.SG_PARTIDO], line=dict(color="white", width=0.6)),
    text=ch_party.seats, textposition="outside",
    customdata=np.stack([ch_party.federation, ch_party.share.map(lambda v: f"{v:.1f}%")], axis=-1),
    hovertemplate="<b>%{y}</b><br>%{customdata[0]}<br>%{x} seats (%{customdata[1]})<extra></extra>"))
fig.update_xaxes(title="Chamber seats")
fig.update_yaxes(autorange="reversed")
fig.update_layout(margin=dict(l=135, r=45, t=80, b=60))
elib.finish(fig, "Chamber of Deputies — 513 seats by party",
            "Federation labels pool the parties that competed together: PT/PCdoB/PV = 88, UNIÃO/PP = 87.", height=560)
elib.save(fig, "n2_chamber_party")
fig.show()

In [4]:
fed_seats = ch_party.groupby("federation", as_index=False).agg(seats=("seats", "sum")).sort_values("seats", ascending=False)
fed_seats["share"] = 100 * fed_seats.seats / fed_seats.seats.sum()

sen_party = sen_win.groupby("SG_PARTIDO", as_index=False).size().rename(columns={"size": "senate"})
gov_w = gov[gov.rk == 1].copy()
gov_party = gov_w.groupby("SG_PARTIDO", as_index=False).size().rename(columns={"size": "governors"})

overs = fed_seats.merge(sen_party, left_on="federation", right_on="SG_PARTIDO", how="left")
overs = overs.merge(gov_party, left_on="federation", right_on="SG_PARTIDO", how="left", suffixes=("", "_gov"))
overs = overs[["federation", "seats", "share", "senate", "governors"]].fillna(0)
overs[["senate", "governors"]] = overs[["senate", "governors"]].astype(int)
overs = overs.rename(columns={"seats": "chamber"})
display(HTML(elib.table(overs.assign(**{"Seat %": overs.share.map(lambda v: f"{v:.1f}")})[["federation", "chamber", "Seat %", "senate", "governors"]]
                        .rename(columns={"federation": "Federation / party", "chamber": "Chamber", "senate": "Senate", "governors": "Governors"}),
                        title="Seats by federation — Chamber, Senate and governorships")))

fig = go.Figure()
fig.add_trace(go.Bar(name="Chamber", x=overs.federation, y=overs.chamber,
                     marker_color=[elib.party_color(f.split("/")[0]) for f in overs.federation]))
fig.add_trace(go.Scatter(name="Senate (2026 class)", x=overs.federation, y=overs.senate, mode="markers",
                         marker=dict(size=14, color="#111827", symbol="diamond"),
                         hovertemplate="<b>%{x}</b><br>%{y} senators<extra></extra>"))
fig.update_yaxes(title="Seats")
fig.update_xaxes(tickangle=-40)
fig.update_layout(margin=dict(l=60, r=10, t=80, b=170))
elib.finish(fig, "Seats by federation — Chamber bars, Senate diamonds",
            "The PL leads both houses; the PT federation is second in the Chamber but only fourth in the Senate.", height=520)
fig.show()

Federation / party,Chamber,Seat %,Senate,Governors
PL,121,23.6,19,6
PT/PCdoB/PV,88,17.2,0,0
UNIÃO/PP,87,17.0,0,0
PSD,43,8.4,2,5
REPUBLICANOS,41,8.0,2,4
MDB,36,7.0,7,1
PODE,27,5.3,2,1
PSB,15,2.9,3,0
PSOL/REDE,15,2.9,0,0
PSDB/Cidadania,11,2.1,0,0


The asymmetry is the story of the 2026 legislature. The PL won **121 Chamber seats (23.6%)** and **19 of the 54 Senate seats**; the PT/PCdoB/PV federation won **88 Chamber seats (17.2%)** but only **6 Senate seats**, because Senate races are majoritarian and statewide — a terrain where the right's statewide tickets performed much better than its proportional lists. The PSD (43 deputies, 5 governors) and the UNIÃO/PP federation (87 deputies) are the pivot blocs.

In [5]:
gmap = gov_w.merge(turn[["SG_UF", "part"]], on="SG_UF")
gmap["pct"] = gmap.pct.round(1)
fig = elib.uf_map(gmap, "SG_PARTIDO", categorical=True, cbar="Party",
                  title="Governors elected in 2026 — by party",
                  subtitle="Six states needed a runoff (AC, AM, DF, ES, RN, TO); the map shows the final winner.",
                  height=620)
fig.update_traces(customdata=np.stack([gmap.SG_UF, gmap.NM_URNA_CANDIDATO.map(elib.pretty_cand), gmap.SG_PARTIDO,
                                       gmap.pct.map(lambda v: f"{v:.1f}%"),
                                       gmap.NR_TURNO.map(lambda v: f"{int(v)} round")], axis=-1))
fig.update_traces(hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]} (%{customdata[2]})"
                                "<br>%{customdata[3]} — %{customdata[4]}<extra></extra>")
elib.save(fig, "n2_governors_map")
fig.show()

In [6]:
sen_state = sen_win.sort_values(["SG_UF", "votos"], ascending=[True, False]).groupby("SG_UF").agg(
    first_party=("SG_PARTIDO", "first"), second_party=("SG_PARTIDO", "last")).reset_index()

fig = elib.uf_map(sen_state, "first_party", categorical=True, cbar="Party of the top senator",
                  title="Senate 2026 — party of the best-voted senator in each state",
                  subtitle="54 seats renewed: two per state. Hover for both winners.",
                  height=620)
fig.update_traces(customdata=np.stack([sen_state.SG_UF, sen_state.first_party, sen_state.second_party], axis=-1))
fig.update_traces(hovertemplate="<b>%{customdata[0]}</b><br>Top senator: %{customdata[1]}<br>Second: %{customdata[2]}<extra></extra>")
elib.save(fig, "n2_senate_map")
fig.show()

## 2. The Senate — 54 seats, two per state

The Senate class of 2026 is the largest renewal of the upper house in a generation: two-thirds of the 81 seats were on the ballot, which is why every state elected a **pair** of senators on the same day. The result is a chamber where the PL holds 19 of the 54 seats up for grabs — more than a third — while the PT holds 6.

In [7]:
sw = sen_win.copy()
sw["State"] = sw.SG_UF + " — " + sw.SG_UF.map(elib.UF_NAME)
sw["Senator"] = sw.NM_URNA_CANDIDATO.map(elib.pretty_cand)
sw["Votes"] = sw.votos.map(elib.fmt_int)
sw["Share"] = sw.share.map(lambda v: f"{v:.2f}")
out = sw.sort_values("votos", ascending=False)[["State", "Senator", "SG_PARTIDO", "Votes", "Share"]]
out.columns = ["State", "Senator", "Party", "Votes", "% of valid Senate votes"]
display(HTML(elib.table(out, title="The 54 senators elected in 2026", note="Shares are of valid Senate votes in the state; each voter casts two votes.", max_rows=60)))

State,Senator,Party,Votes,% of valid Senate votes
SP — São Paulo,Guilherme Derrite,PP,"13,273,880",30.59
SP — São Paulo,André Do Prado,PL,"12,703,089",29.28
MG — Minas Gerais,Domingos Sávio,PL,"4,968,829",24.21
BA — Bahia,Rui Costa,PT,"4,335,196",30.07
RJ — Rio de Janeiro,Carlos Portinho,PL,"4,264,932",26.77
MG — Minas Gerais,Marília Campos,PT,"3,990,864",19.44
BA — Bahia,Jaques Wagner,PT,"3,986,690",27.65
RJ — Rio de Janeiro,Carlos Jordy,PL,"3,912,405",24.56
RS — Rio Grande do Sul,Sanderson,PL,"3,453,316",30.11
RS — Rio Grande do Sul,Marcel Van Hattem,NOVO,"3,449,053",30.08


In [8]:
pair = sen_win.groupby("SG_UF").agg(parties=("SG_PARTIDO", lambda s: tuple(sorted(s)))).reset_index()
pair["same"] = pair.parties.map(lambda t: len(set(t)) == 1)
same = int(pair["same"].sum())

conc = sen_win.groupby("SG_UF").votos.apply(lambda s: 100 * s.max() / s.sum())
fig = go.Figure()
fig.add_trace(go.Histogram(x=conc, nbinsx=20, marker_color="#1d4e89"))
fig.update_xaxes(title="Top senator's share of the state's valid Senate votes (%)")
fig.update_yaxes(title="States")
elib.finish(fig, "How concentrated was the Senate vote?",
            f"In {same} of 27 states the two elected senators came from the same party. Median top share: {conc.median():.1f}%.", height=380)
fig.show()

top = sen_win.nlargest(10, "votos")[["SG_UF", "NM_URNA_CANDIDATO", "SG_PARTIDO", "votos", "share"]]
top = top.assign(Senator=top.NM_URNA_CANDIDATO.map(elib.pretty_cand), State=top.SG_UF,
                 Votes=top.votos.map(elib.fmt_int), Share=top.share.map(lambda v: f"{v:.1f}"))[["State", "Senator", "SG_PARTIDO", "Votes", "Share"]]
top.columns = ["State", "Senator", "Party", "Votes", "Share %"]
display(HTML(elib.table(top, title="The ten best-voted senators in the country")))

State,Senator,Party,Votes,Share %
SP,Guilherme Derrite,PP,"13,273,880",30.6
SP,André Do Prado,PL,"12,703,089",29.3
MG,Domingos Sávio,PL,"4,968,829",24.2
BA,Rui Costa,PT,"4,335,196",30.1
RJ,Carlos Portinho,PL,"4,264,932",26.8
MG,Marília Campos,PT,"3,990,864",19.4
BA,Jaques Wagner,PT,"3,986,690",27.6
RJ,Carlos Jordy,PL,"3,912,405",24.6
RS,Sanderson,PL,"3,453,316",30.1
RS,Marcel Van Hattem,NOVO,"3,449,053",30.1


**Reading the Senate.** Two structural facts explain the outcome. First, **statewide majoritarianism favours known names**: the median top-senator share was 29.1% of the valid Senate vote and **no senator in the country reached 36%** — the two-seat race fragments the vote even for the strongest names. Second, **ticket-splitting is the rule**: in **21 of the 27 states** the two elected senators came from different parties — voters routinely split their two Senate votes — but the aggregate still produced a 19-seat PL plurality.

The Senate is therefore the chamber where the government's legislative position is weakest: 19 PL senators against 6 PT senators, with the PSD (2), MDB (7), UNIÃO (3) and PP (3) holding the balance.

## 3. The Chamber and the state assemblies

Chamber delegations range from **8 deputies** in the smallest states to **70 in São Paulo**, fixed by the complementary law that caps the chamber at 513 and sets a minimum of 8 and a maximum of 70 per state. That design — combined with state-level district magnitudes — is the main source of Brazil's disproportionality, and it is visible in the map below.

In [9]:
ch_state = ch_win.groupby("SG_UF", as_index=False).size().rename(columns={"size": "seats"})
ch_state = ch_state.merge(turn[["SG_UF", "aptos", "part"]], on="SG_UF")
ch_state["seats_per_100k"] = 100000 * ch_state.seats / ch_state.aptos

fig = elib.uf_map(ch_state, "seats", colorscale="Blues", cbar="Deputies",
                  title="Chamber delegation size by state",
                  subtitle="From 8 deputies in the smallest states to 70 in São Paulo — the 513-seat cap in action.",
                  height=620)
fig.update_traces(customdata=np.stack([ch_state.SG_UF, ch_state.seats.map(lambda v: f"{v}"),
                                       ch_state.aptos.map(elib.fmt_int),
                                       ch_state.seats_per_100k.map(lambda v: f"{v:.2f}")], axis=-1))
fig.update_traces(hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]} deputies<br>Electorate: %{customdata[2]}"
                                "<br>%{customdata[3]} deputies per 100k voters<extra></extra>")
elib.save(fig, "n2_chamber_seats_state")
fig.show()

fig = go.Figure(go.Scatter(x=ch_state.aptos, y=ch_state.seats, mode="markers+text",
                           text=ch_state.SG_UF, textposition="top center", textfont=dict(size=9),
                           marker=dict(size=9, color="#1d4e89")))
fig.update_xaxes(type="log", title="Registered voters (log)")
fig.update_yaxes(title="Chamber seats")
elib.finish(fig, "Delegation size is not proportional to population",
            "The floor of 8 seats gives small states a structural premium; São Paulo has one deputy for every ~487k voters.", height=420)
fig.show()

In [10]:
piv = ch_win.pivot_table(index="SG_UF", columns="SG_PARTIDO", values="SQ_CANDIDATO", aggfunc="count").fillna(0)
piv = piv.reindex(elib.UF_ORDER).fillna(0)
piv = piv[piv.sum().sort_values(ascending=False).index]

fig = go.Figure(go.Heatmap(
    z=piv.values, x=piv.columns, y=piv.index, colorscale="Blues", zmin=0, zmax=4,
    colorbar=dict(title="Seats", thickness=12),
    hovertemplate="<b>%{y}</b><br>%{x}: %{z:.0f} seats<extra></extra>"))
elib.finish(fig, "Party composition of each state's Chamber delegation",
            "Columns ordered by national size. The PL is the largest party in 13 of the 27 states; the MDB leads in 4 and the PT in 2.", height=720)
fig.show()

**Vote-to-seat conversion.** The chart below computes, for each state, the Gallagher index of the Chamber election — how far the seat shares deviate from the nominal vote shares. Small states are systematically more disproportional, because a handful of seats makes the quotient threshold brutal for small parties.

In [11]:
rows = []
for uf, g in ch[ch.SG_UF != "ZZ"].groupby("SG_UF"):
    votes = g.groupby("SG_PARTIDO").votos.sum()
    seats = ch_win[ch_win.SG_UF == uf].groupby("SG_PARTIDO").size()
    parties = votes.index.union(seats.index)
    v = (votes.reindex(parties).fillna(0) / votes.sum())
    s = (seats.reindex(parties).fillna(0) / max(seats.sum(), 1))
    rows.append({"SG_UF": uf, "seats": int(seats.sum()), "gallagher": 100 * np.sqrt(0.5 * ((v - s) ** 2).sum()),
                 "enp_v": 1 / (v[v > 0] ** 2).sum(), "enp_s": 1 / (s[s > 0] ** 2).sum()})
gal = pd.DataFrame(rows)
gal = elib.add_uf_meta(gal).sort_values("gallagher")

fig = go.Figure(go.Bar(x=gal.SG_UF, y=gal.gallagher, marker_color=gal.REGIAO.map(elib.REGION_COLORS),
                       customdata=np.stack([gal.REGIAO_NOME, gal.seats], axis=-1),
                       hovertemplate="<b>%{x}</b><br>%{customdata[0]}<br>%{customdata[1]} seats<br>Gallagher: %{y:.1f}<extra></extra>"))
fig.update_yaxes(title="Gallagher index (seat vs vote shares)")
elib.finish(fig, "Disproportionality by state — Chamber election",
            "Coloured by region. The national index is 3.0; small northern states run two to three times higher.", height=420)
fig.show()

display(HTML(elib.table(
    gal.assign(State=gal.SG_UF + " — " + gal.UF_NOME, Seats=gal.seats,
               **{"Gallagher": gal.gallagher.map(lambda v: f"{v:.1f}"),
                  "ENP votes": gal.enp_v.map(lambda v: f"{v:.1f}"),
                  "ENP seats": gal.enp_s.map(lambda v: f"{v:.1f}")})[["State", "Seats", "Gallagher", "ENP votes", "ENP seats"]],
    title="Seat/vote distortion by state", note="ENP = effective number of parties (Laakso-Taagepera).")))

State,Seats,Gallagher,ENP votes,ENP seats
PE — Pernambuco,25,3.3,11.4,10.6
MG — Minas Gerais,53,3.7,4.6,4.5
BA — Bahia,39,3.8,9.8,8.7
PI — Piauí,10,4.4,5.9,5.0
PR — Paraná,30,4.8,9.4,8.0
RS — Rio Grande do Sul,31,5.1,9.0,7.3
SP — São Paulo,70,5.4,7.9,7.4
RJ — Rio de Janeiro,46,6.0,8.4,6.2
AL — Alagoas,9,6.3,5.0,4.3
CE — Ceará,22,7.0,9.2,7.8


In [12]:
top_ch = ch.nlargest(15, "votos")[["SG_UF", "NM_URNA_CANDIDATO", "SG_PARTIDO", "votos", "situacao"]].copy()
party_votes = ch.groupby(["SG_UF", "SG_PARTIDO"]).votos.sum().rename("party_total").reset_index()
top_ch = top_ch.merge(party_votes, on=["SG_UF", "SG_PARTIDO"], how="left")
top_ch["share_party"] = 100 * top_ch.votos / top_ch.party_total
display(HTML(elib.table(
    top_ch.assign(State=top_ch.SG_UF, Candidate=top_ch.NM_URNA_CANDIDATO.map(elib.pretty_cand),
                  Votes=top_ch.votos.map(elib.fmt_int), Situation=top_ch.situacao,
                  **{"% of own party in the state": top_ch.share_party.map(lambda v: f"{v:.1f}")})[
        ["State", "Candidate", "SG_PARTIDO", "Votes", "% of own party in the state", "Situation"]]
        .rename(columns={"SG_PARTIDO": "Party"}),
    title="The 15 best-voted deputies in the country",
    note="The last column shows how much of the candidate's own party vote in that state came from this one person — the puxador de voto effect.")))

State,Candidate,Party,Votes,% of own party in the state,Situation
MG,Nikolas Ferreira,PL,"3,119,318",68.4,ELEITO POR QP
SP,Lucas Pavanato,PL,"3,038,438",48.4,ELEITO POR QP
SP,Erika Hilton,PSOL,"1,596,472",53.6,ELEITO POR QP
CE,André Fernandes,PL,"684,372",62.8,ELEITO POR QP
SP,Sâmia Bomfim,PSOL,"583,107",19.6,ELEITO POR QP
SC,Julia Zanatta,PL,"560,223",37.3,ELEITO POR QP
MG,Ana Elisa,PT,"524,219",25.6,ELEITO POR QP
SP,Kim Kataguiri,MISSÃO,"520,071",81.2,ELEITO POR QP
SP,Tabata Amaral,PSB,"476,383",50.6,ELEITO POR QP
PR,Jeffrey Chiquini,NOVO,"425,399",45.9,ELEITO POR QP


**The vote-pullers.** Brazil's open-list system rewards individual vote concentration, and 2026 was no exception: the most concentrated case is a deputy who took **79.5% of his own party's vote in his state**, and several others passed a third. In small states a single candidate can drag a whole slate over the quotient threshold — which is why the puxador de voto remains the central strategic figure of Brazilian proportional politics.

In the state assemblies the pattern repeats at a smaller scale: **1,059 seats**, of which the PL won 212 and the PT 141.

In [13]:
asm_party = asm_win.groupby("SG_PARTIDO", as_index=False).size().rename(columns={"size": "seats"}).sort_values("seats", ascending=False)

fig = go.Figure(go.Bar(x=asm_party.SG_PARTIDO, y=asm_party.seats,
                       marker_color=[elib.party_color(p) for p in asm_party.SG_PARTIDO],
                       text=asm_party.seats, textposition="outside"))
fig.update_yaxes(title="State/district seats")
fig.update_xaxes(tickangle=-45)
fig.update_layout(margin=dict(l=115, r=10, t=80, b=150))
elib.finish(fig, f"State assemblies — {int(asm_party.seats.sum()):,} seats by party",
            "The PL is the largest party in the state legislatures too, but the PT is relatively stronger there than in the Chamber.", height=520)
fig.show()

apiv = asm_win.pivot_table(index="SG_UF", columns="SG_PARTIDO", values="SQ_CANDIDATO", aggfunc="count").fillna(0)
apiv = apiv.reindex(elib.UF_ORDER).fillna(0)
apiv = apiv[apiv.sum().sort_values(ascending=False).index]

fig = go.Figure(go.Heatmap(z=apiv.values, x=apiv.columns, y=apiv.index,
                           colorscale="Reds", zmin=0, zmax=10,
                           colorbar=dict(title="Seats", thickness=12),
                           hovertemplate="<b>%{y}</b><br>%{x}: %{z:.0f} seats<extra></extra>"))
elib.finish(fig, "Party composition of each state assembly",
            "Rows are states, columns are parties. The PL and the PT dominate the two ends of the spectrum.", height=720)
fig.show()

## 4. State dossiers — all 27 federal units

Each block below reports the final governor result, the two senators elected, the Chamber delegation, the state assembly, turnout, and an **interactive municipal map of the governor race** (hover for the winner, the share and the margin in each municipality).

For states decided in a runoff the map and the vote shares refer to the second round; otherwise to the first.

In [14]:
gov_mun = con.execute(r'''
WITH g AS (
  SELECT SG_UF, CD_IBGE, NOME_IBGE, NR_TURNO, SQ_CANDIDATO, NM_CANDIDATO, NM_URNA_CANDIDATO, SG_PARTIDO, sum(VOTOS) v
  FROM cand WHERE DS_CARGO = 'Governador' AND CD_IBGE IS NOT NULL GROUP BY ALL),
t AS (SELECT SG_UF, CD_IBGE, NR_TURNO, sum(v) tot FROM g GROUP BY 1, 2, 3),
r AS (SELECT g.*, t.tot, 100.0 * g.v / t.tot pct,
             row_number() OVER (PARTITION BY SG_UF, CD_IBGE, NR_TURNO ORDER BY g.v DESC) rk
      FROM g JOIN t USING (SG_UF, CD_IBGE, NR_TURNO)),
w AS (SELECT SG_UF, max(NR_TURNO) turno FROM g GROUP BY 1)
SELECT r.* FROM r JOIN w ON r.SG_UF = w.SG_UF AND r.NR_TURNO = w.turno
''').df()

mturn = con.execute(r'''
SELECT CD_IBGE, sum(QT_APTOS) aptos, sum(QT_COMPARECIMENTO) comp,
       100.0 * sum(QT_COMPARECIMENTO) / sum(QT_APTOS) turnout
FROM det WHERE CD_CARGO = 3 AND NR_TURNO = 1 GROUP BY 1
''').df()

print("municipal governor rows:", len(gov_mun), "| states:", gov_mun.SG_UF.nunique())

municipal governor rows: 38672 | states: 27


In [15]:
for uf in elib.UF_ORDER:
    g = gov_mun[gov_mun.SG_UF == uf].sort_values("rk")
    winner = gov[(gov.SG_UF == uf) & (gov.rk == 1)].iloc[0]
    runner = gov[(gov.SG_UF == uf) & (gov.rk == 2)].iloc[0]
    suffix = {1: "st", 2: "nd"}.get(int(winner.NR_TURNO), "th")
    sens = sen_win[sen_win.SG_UF == uf].sort_values("votos", ascending=False)
    cham = ch_win[ch_win.SG_UF == uf]
    asm_s = asm_win[asm_win.SG_UF == uf]
    t = turn[turn.SG_UF == uf].iloc[0]

    display(Markdown(f"### {uf} — {elib.UF_NAME[uf]} · {elib.REGIONS[elib.UF_REGION[uf]]} · {int(t.aptos):,} voters"))

    cards = [
        ("Governor", elib.pretty_cand(winner.NM_URNA_CANDIDATO) + f" ({winner.SG_PARTIDO})",
         f"{winner.pct:.1f}% in the {int(winner.NR_TURNO)}{suffix} round"),
        ("Runner-up", elib.pretty_cand(runner.NM_URNA_CANDIDATO) + f" ({runner.SG_PARTIDO})",
         f"{runner.pct:.1f}% · margin {winner.pct - runner.pct:+.1f} p.p."),
        ("Senators elected", ", ".join(sens.SG_PARTIDO),
         " · ".join(elib.pretty_cand(n) for n in sens.NM_URNA_CANDIDATO)),
        ("Chamber / Assembly", f"{len(cham)} / {len(asm_s)}", "deputies / state deputies"),
        ("Turnout", f"{t.part:.1f}%", f"{int(t.comp):,} of {int(t.aptos):,}"),
    ]
    display(HTML(elib.kpis(cards, cols=5, small=True)))

    cp = cham.groupby("SG_PARTIDO").size().sort_values(ascending=False)
    ap = asm_s.groupby("SG_PARTIDO").size().sort_values(ascending=False)
    allp = list(dict.fromkeys(list(cp.index) + list(ap.index)))
    tbl = pd.DataFrame({
        "Party": allp,
        "Chamber": [cp.get(p, "") for p in allp],
        "Assembly": [ap.get(p, "") for p in allp],
    })
    if len(tbl):
        display(HTML(elib.table(tbl, title="Seats won in this state")))

    win2 = g[g.rk == 1][["CD_IBGE", "NM_URNA_CANDIDATO", "SG_PARTIDO", "pct", "v"]].copy()
    win2["CD_IBGE"] = win2.CD_IBGE.astype(str)
    run2 = g[g.rk == 2][["CD_IBGE", "NOME_IBGE", "NM_URNA_CANDIDATO", "SG_PARTIDO", "pct"]].copy()
    run2["CD_IBGE"] = run2.CD_IBGE.astype(str)
    d = win2.merge(run2, on="CD_IBGE", suffixes=("", "_run")).merge(mturn, on="CD_IBGE", how="left")
    d["margin"] = d.pct - d.pct_run
    d["winner"] = d.NM_URNA_CANDIDATO.map(elib.pretty_cand)
    d["runner"] = d.NM_URNA_CANDIDATO_run.map(elib.pretty_cand)

    fig = elib.mun_map(uf, d, "SG_PARTIDO", categorical=True, cbar="Winner",
                       title=f"{elib.UF_NAME[uf]} — governor race by municipality",
                       subtitle=f"{int(winner.NR_TURNO)}{suffix} round · {elib.pretty_cand(winner.NM_URNA_CANDIDATO)} ({winner.SG_PARTIDO}) {winner.pct:.1f}% vs {elib.pretty_cand(runner.NM_URNA_CANDIDATO)} ({runner.SG_PARTIDO}) {runner.pct:.1f}%",
                       height=460)
    fig.update_traces(customdata=np.stack([d.NOME_IBGE.str.title(), d.winner, d.SG_PARTIDO,
                                           d.pct.map(lambda v: f"{v:.1f}%"), d.runner,
                                           d.margin.map(lambda v: f"{v:+.1f} p.p."),
                                           d.turnout.fillna(0).map(lambda v: f"{v:.1f}%")], axis=-1))
    fig.update_traces(hovertemplate="<b>%{customdata[0]}</b><br>Winner: %{customdata[1]} (%{customdata[2]}) %{customdata[3]}"
                                    "<br>Runner-up: %{customdata[4]}<br>Margin: %{customdata[5]}<br>Turnout: %{customdata[6]}<extra></extra>")
    fig.show()

### AC — Acre · Norte · 613,316 voters

Party,Chamber,Assembly
PP,2,5
UNIÃO,2,4
MDB,1,2
PCDOB,1,1
PL,1,4
REPUBLICANOS,1,3
PSD,,2
NOVO,,1
PDT,,1
PSDB,,1


### AL — Alagoas · Nordeste · 2,439,190 voters

Party,Chamber,Assembly
PP,3,3
MDB,2,15
PSD,2,
PSDB,1,2
UNIÃO,1,2
PT,,3
PL,,2


### AP — Amapá · Norte · 576,359 voters

...

Party,Chamber,Assembly
PL,19,27
PT,11,16
PODE,6,6
PSD,6,9
PSOL,6,9
REPUBLICANOS,6,6
MDB,4,5
PP,3,5
NOVO,2,1
PSB,2,3


### SE — Sergipe · Nordeste · 1,737,460 voters

Party,Chamber,Assembly
PSD,2,5
UNIÃO,2,6
PP,1,
PSB,1,2
PT,1,1
REPUBLICANOS,1,3
MDB,,2
PL,,2
PV,,2
AVANTE,,1


### TO — Tocantins · Norte · 1,180,433 voters

Party,Chamber,Assembly
REPUBLICANOS,3,5
PL,1,5
PODE,1,2
PP,1,
PSDB,1,2
UNIÃO,1,3
MDB,,3
PCDOB,,1
PRD,,1
PSD,,1


## 5. Cross-cutting patterns

Three patterns cut across the 27 dossiers.

**Governors do not follow the presidential map.** The PL won the three southern states, but the PT took Bahia, Ceará, Piauí and Rio Grande do Norte, and the PSD took Amazonas, Amapá, Maranhão, Pernambuco and Sergipe. Gubernatorial coalitions are built locally and often include parties from both national blocs.

**The runoff mattered in six states.** AC, AM, DF, ES, RN and TO were decided in the second round, and in two of them (RN and TO) the first-round leader lost — a reminder that the runoff is a separate election with its own dynamics.

**Turnout and invalid votes are different phenomena.** States with the highest null-vote rates (Sergipe 10.8%, Paraíba 9.8%, Rio de Janeiro 9.7%) are not the states with the lowest turnout — invalidity expresses political rejection rather than administrative failure.

In [16]:
pres_st = con.execute(r'''
WITH s AS (SELECT SG_UF, NM_VOTAVEL, sum(VOTOS) v FROM pres
           WHERE NR_TURNO = 2 AND SG_UF <> 'ZZ' GROUP BY 1, 2)
SELECT SG_UF, arg_max(NM_VOTAVEL, v) pres_winner FROM s GROUP BY 1
''').df()
pres_st["pres_party"] = pres_st.pres_winner.map(lambda n: "PT" if n.startswith("LUIZ") else "PL")
align = gov_w[["SG_UF", "SG_PARTIDO", "NM_URNA_CANDIDATO", "pct"]].merge(pres_st, on="SG_UF")
align["aligned"] = align.SG_PARTIDO == align.pres_party
print("governors from the presidential winner's party:", int(align.aligned.sum()), "of 27")

tbl = align.assign(Governor=align.NM_URNA_CANDIDATO.map(elib.pretty_cand),
                   **{"Gov. party": align.SG_PARTIDO, "Pres. winner": align.pres_party,
                      "Gov. %": align.pct.map(lambda v: f"{v:.1f}")})[["SG_UF", "Governor", "Gov. party", "Gov. %", "Pres. winner"]]
tbl["Aligned"] = np.where(align.aligned, "yes", "no")
display(HTML(elib.table(tbl, title="Governor versus presidential winner, state by state")))

governors from the presidential winner's party: 10 of 27


SG_UF,Governor,Gov. party,Gov. %,Pres. winner,Aligned
AP,DR. Furlan,PSD,62.8,PT,no
ES,Lorenzo Pazolini,REPUBLICANOS,52.2,PL,no
MA,Eduardo Braide,PSD,54.0,PT,no
MG,Cleitinho Azevedo,REPUBLICANOS,55.4,PT,no
PE,Raquel Lyra,PSD,53.3,PT,no
MT,Otaviano Pivetta,REPUBLICANOS,60.8,PL,no
PB,Lucas Ribeiro,PP,64.3,PT,no
BA,Jerônimo Rodrigues,PT,55.8,PT,yes
MS,Eduardo Riedel,PP,67.1,PL,no
SE,Fábio,PSD,59.3,PT,no


## 6. What the legislative map means for 2027-2030

1. **A right-of-centre Congress with a left-of-centre president.** The PL (121), UNIÃO/PP (87), PSD (43), Republicanos (41), MDB (36) and PODE (27) hold 355 Chamber seats; the PT federation holds 88. Any government bill needs at least two of those centre-right parties.
2. **The Senate is the bottleneck.** With 19 PL senators in the 2026 class and only 6 PT, the government cannot rely on the upper house for structural reforms. The PSD (2) and MDB (7) delegations are, again, the pivot.
3. **State power is fragmented.** Eight parties govern the 27 states: PL 6, PSD 5, PP 4, PT 4, Republicanos 4, PSDB 2, MDB 1, PODE 1. That fragmentation is a resource for governors negotiating with Brasília and a constraint for any national coordinator.
4. **The federation experiment worked.** The five federations elected 209 Chamber seats and 87 state-assembly seats between them; the PT/PCdoB/PV federation is now the second largest bloc in the Chamber and the largest single organisation of the left.

### Caveats
* Chamber and assembly vote shares use nominal votes only; party-list votes (3.4 M and 7.2 M) are excluded.
* Seat totals attribute each deputy to his/her party label, even when the seat was won through a federation's pooled quotient.
* The 27 Senate seats elected in 2022 are not part of this dataset; the 54 seats analysed here are the 2026 class only.
* Municipality-level governor maps use the deciding round and exclude voters abroad.